# 🏠 RAGScore + Ollama — Free RAG Evaluation on Colab

Evaluate your RAG **completely free** — Ollama runs inside Colab, no API keys needed.

| | |
|---|---|
| 🔒 **Privacy** | No external API calls — LLM runs inside Colab |
| 💰 **Cost** | Free — uses Colab's GPU |
| 🤖 **Model** | `qwen2.5:1.5b` (fast, fits Colab free tier) |
| ⏱️ **Setup** | ~2 min install, then evaluate in minutes |

**Just run all cells in order — no configuration needed!**

## Step 1: Install Ollama + RAGScore

> ⏳ This takes ~2 minutes. It installs Ollama, pulls `qwen2.5:1.5b` (~900 MB), and starts the server.

> **💡 Model quality note:** `qwen2.5:1.5b` is used here to demonstrate the pipeline on Colab's free GPU — it's fast but small. For production-quality evaluation, use a larger model:
>
> | Model | Size | Where to run |
> |-------|------|-------------|
> | `qwen2.5:1.5b` | 0.9 GB | ✅ Colab free tier (this demo) |
> | `llama3.2:3b` | 2.0 GB | Colab free / local |
> | `llama3.1:8b` | 4.7 GB | Colab Pro / local 8GB RAM |
> | `qwen2.5:14b` | 9.0 GB | Local 16GB RAM |
> | `llama3.1:70b` | 40 GB | Local 64GB RAM / Colab A100 |
>
> To use a different model locally, replace `qwen2.5:1.5b` with your model name after `ollama pull <model>`.

In [ ]:
import subprocess, time, shutil

# Install zstd (required by Ollama installer) and RAGScore
print("📦 Installing dependencies...")
subprocess.run(["apt-get", "install", "-y", "-q", "zstd"], check=True)
subprocess.run(["pip", "install", "-q", "ragscore[notebook]>=0.7.7", "numpy"], check=True)

# Install Ollama
print("📦 Installing Ollama...")
result = subprocess.run(
    "curl -fsSL https://ollama.ai/install.sh | sh",
    shell=True, capture_output=True, text=True
)
if result.returncode != 0:
    print(result.stderr[-2000:])
    raise RuntimeError("Ollama install failed")

# Start Ollama server in background
ollama_bin = shutil.which("ollama") or "/usr/local/bin/ollama"
print(f"🚀 Starting Ollama server...")
subprocess.Popen([ollama_bin, "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)

# Pull model
print("📥 Pulling qwen2.5:1.5b (~900 MB)...")
subprocess.run([ollama_bin, "pull", "qwen2.5:1.5b"], check=True)

# Patch asyncio for notebook compatibility
import nest_asyncio
nest_asyncio.apply()

OLLAMA_MODEL = "qwen2.5:1.5b"
print(f"\n✅ Ready! Ollama running with {OLLAMA_MODEL}")

## Step 2: Create a sample document

In [ ]:
%%writefile sample_doc.txt
Retrieval-Augmented Generation (RAG) is an AI architecture that enhances large language models by combining them with external knowledge retrieval. Instead of relying solely on the model's training data, RAG systems first search a knowledge base for relevant documents, then use these documents as context when generating responses.

The RAG pipeline consists of three main stages: indexing, retrieval, and generation. During indexing, documents are split into chunks, converted to vector embeddings, and stored in a vector database. Popular vector databases include Pinecone, Weaviate, Qdrant, ChromaDB, and pgvector for PostgreSQL. The choice of embedding model significantly impacts retrieval quality — OpenAI's text-embedding-3-small and Cohere's embed-v3 are popular choices.

Retrieval is triggered when a user submits a query. The query is embedded using the same model, and the vector database returns the top-k most similar chunks based on cosine similarity or other distance metrics. Hybrid retrieval combines vector search with traditional keyword search (BM25) for better results. Re-ranking models like Cohere Rerank or cross-encoders can further improve relevance.

The generation stage feeds the retrieved context along with the original question to an LLM. The system prompt typically instructs the model to answer based only on the provided context, reducing hallucination. Common LLMs for RAG include GPT-4o, Claude 3.5, Llama 3.1, and Mistral. Temperature settings between 0.1 and 0.3 are recommended for factual QA.

Chunking strategy is critical for RAG performance. Fixed-size chunking (e.g., 512 tokens with 50-token overlap) is simple but may split sentences awkwardly. Semantic chunking groups related sentences together, while recursive character splitting offers a balance. Document-specific chunking can leverage headers and sections in structured documents.

Evaluating RAG systems requires measuring both retrieval quality and generation quality. Retrieval metrics include precision@k, recall@k, and NDCG. Generation metrics include faithfulness (does the answer match the source?), relevance (does it answer the question?), and completeness (does it cover all key points?). RAGScore automates this evaluation using LLM-as-judge methodology.

## Step 3: Build a mini RAG with Ollama

In [ ]:
import numpy as np
import urllib.request, json

with open("sample_doc.txt") as f:
    text = f.read()
chunks = [p.strip() for p in text.split("\n\n") if len(p.strip()) > 50]
print(f"✅ {len(chunks)} chunks created")

def ollama_embed(text):
    data = json.dumps({"model": OLLAMA_MODEL, "input": text}).encode()
    req = urllib.request.Request(
        "http://localhost:11434/api/embed",
        data=data, headers={"Content-Type": "application/json"},
    )
    return json.loads(urllib.request.urlopen(req).read())["embeddings"][0]

def ollama_chat(messages):
    data = json.dumps({
        "model": OLLAMA_MODEL, "messages": messages,
        "stream": False, "options": {"temperature": 0.3},
    }).encode()
    req = urllib.request.Request(
        "http://localhost:11434/api/chat",
        data=data, headers={"Content-Type": "application/json"},
    )
    return json.loads(urllib.request.urlopen(req).read())["message"]["content"]

print("Embedding chunks...")
chunk_embeddings = np.array([ollama_embed(c) for c in chunks])
print(f"✅ {len(chunk_embeddings)} chunks embedded")

def my_rag(question):
    q_emb = np.array(ollama_embed(question))
    sims = chunk_embeddings @ q_emb / (
        np.linalg.norm(chunk_embeddings, axis=1) * np.linalg.norm(q_emb)
    )
    top_idx = np.argsort(sims)[-3:][::-1]
    context = "\n\n".join([chunks[i] for i in top_idx])
    return ollama_chat([
        {"role": "system", "content": "Answer based only on the provided context. Be concise."},
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"},
    ])

print("\n🧪 Test:")
print(my_rag("What is RAG?"))

## Step 4: Evaluate RAG with RAGScore (no API key!)

In [ ]:
from ragscore import quick_test

result = quick_test(
    my_rag,
    docs="sample_doc.txt",
    n=5,
    model=OLLAMA_MODEL,        # Ollama generates the QA pairs
    judge_model=OLLAMA_MODEL,  # Ollama judges the answers
)

print(f"\n📊 Accuracy:  {result.accuracy:.0%}")
print(f"📊 Avg Score: {result.avg_score:.1f}/5")
print("\n📋 Questions generated:")
for _, row in result.df.iterrows():
    print(f"  [{row['score']}/5] {row['question'][:80]}")

## Step 5: Detailed 5-metric evaluation

In [ ]:
detailed_result = quick_test(
    my_rag,
    docs="sample_doc.txt",
    n=5,
    model=OLLAMA_MODEL,
    judge_model=OLLAMA_MODEL,
    detailed=True,
)

display(detailed_result.df[[
    "question", "score", "correctness", "completeness",
    "relevance", "conciseness", "faithfulness"
]])

## Step 6: Audience-targeted QA generation

In [ ]:
dev_result = quick_test(
    my_rag, docs="sample_doc.txt", n=3,
    model=OLLAMA_MODEL, judge_model=OLLAMA_MODEL,
    audience="backend developers",
    purpose="building a RAG pipeline",
)
print("👨‍💻 Developer questions:")
for _, row in dev_result.df.iterrows():
    print(f"  [{row['score']}/5] {row['question'][:80]}")

mgr_result = quick_test(
    my_rag, docs="sample_doc.txt", n=3,
    model=OLLAMA_MODEL, judge_model=OLLAMA_MODEL,
    audience="product managers",
    purpose="evaluating RAG vendors",
)
print("\n📋 Manager questions:")
for _, row in mgr_result.df.iterrows():
    print(f"  [{row['score']}/5] {row['question'][:80]}")

---

## 📚 Resources

- **GitHub**: https://github.com/HZYAI/RagScore
- **PyPI**: https://pypi.org/project/ragscore/
- **Ollama**: https://ollama.ai
- **Website**: https://ragscore.io

⭐ If RAGScore helps you, give us a star on GitHub!